In [10]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

# --- Kalman Filter Implementation
def kalman_filter_1d(observations, initial_mean=None, initial_covariance=1.0,
                     observation_covariance=1.0, process_covariance=0.01):
    # ... (Full Kalman filter code) ...
    n_timesteps = len(observations)
    if n_timesteps == 0: return np.array([])
    x_hat, P = np.zeros(n_timesteps), np.zeros(n_timesteps)
    obs_array = np.asarray(observations, dtype=float)
    if initial_mean is None:
        first_valid_idx = np.where(pd.notna(obs_array))[0]
        x_hat[0] = obs_array[first_valid_idx[0]] if len(first_valid_idx) > 0 else 0.0
    else: x_hat[0] = float(initial_mean)
    P[0] = float(initial_covariance)
    for k in range(1, n_timesteps):
        x_hat_minus_k, P_minus_k = x_hat[k-1], P[k-1] + process_covariance
        current_obs = obs_array[k]
        if pd.isna(current_obs): x_hat[k], P[k] = x_hat_minus_k, P_minus_k
        else:
            denom = P_minus_k + observation_covariance
            K_k = P_minus_k / denom if abs(denom) > 1e-9 else 0.0
            x_hat[k] = x_hat_minus_k + K_k * (current_obs - x_hat_minus_k)
            P[k] = (1 - K_k) * P_minus_k
    return x_hat

# --- Trading Strategy (
def sma_trading_strategy(ticker, start_date, end_date, initial_cash,
                         kalman_qr_factor=0.01,
                         macd_short_span=12,
                         macd_long_span=26,
                         macd_signal_span=9,
                         show_plots=True):

    # ... (It calculates everything and returns the dictionary with all metrics) ...
    data = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
    min_data_len = max(20, int(round(macd_long_span)) + int(round(macd_signal_span)) + 15)
    if data.empty or len(data) < min_data_len: return None
    p_macd_short = int(round(macd_short_span)); p_macd_long = int(round(macd_long_span)); p_macd_signal = int(round(macd_signal_span))
    p_macd_short = max(1, p_macd_short); p_macd_long = max(p_macd_short + 1, p_macd_long); p_macd_signal = max(1, p_macd_signal)
    raw_close_prices_col = data['Close']
    if isinstance(raw_close_prices_col, pd.DataFrame):
        if not raw_close_prices_col.empty and raw_close_prices_col.shape[1] == 1: raw_close_prices_col = raw_close_prices_col.iloc[:, 0]
        else: return None
    data_close_series = raw_close_prices_col.squeeze()
    if not isinstance(data_close_series, pd.Series): return None
    close_prices_for_var = data_close_series.dropna().values
    price_diffs_values = np.diff(close_prices_for_var) if len(close_prices_for_var) >= 2 else np.array([])
    obs_noise_var = float(np.var(price_diffs_values, ddof=0)) if len(price_diffs_values) >=1 else 0.1
    if pd.isna(obs_noise_var) or obs_noise_var <= 1e-7: obs_noise_var = 0.1
    proc_noise_var = obs_noise_var * kalman_qr_factor
    if pd.isna(proc_noise_var) or proc_noise_var <= 1e-9: proc_noise_var = 1e-5
    init_state_cov = obs_noise_var
    kf_init_mean = data_close_series.bfill().ffill().iloc[0] if not data_close_series.bfill().ffill().empty else 0.0
    if pd.isna(kf_init_mean): kf_init_mean = 0.0
    kalman_output = kalman_filter_1d(data_close_series, kf_init_mean, init_state_cov, obs_noise_var, proc_noise_var)
    kalman_close_series = pd.Series(kalman_output, index=data_close_series.index).bfill().ffill()
    source_price_for_signals = kalman_close_series
    delta_filt = source_price_for_signals.diff()
    gain_filt = delta_filt.where(delta_filt > 0, 0.0); loss_filt = (-delta_filt).where(delta_filt < 0, 0.0)
    rsi_window = 14
    avg_gain_filt = gain_filt.rolling(window=rsi_window, min_periods=1).mean(); avg_loss_filt = loss_filt.rolling(window=rsi_window, min_periods=1).mean()
    rs_denom_filt = avg_loss_filt.replace(0, 1e-9); rs_filt = avg_gain_filt / rs_denom_filt
    rsi_series_filt = 100.0 - (100.0 / (1.0 + rs_filt))
    rsi_series_filt.loc[rs_denom_filt == 1e-9] = 100.0; rsi_series_filt.loc[(avg_gain_filt == 0) & (rs_denom_filt != 1e-9)] = 0.0; rsi_series_filt.loc[(avg_gain_filt == 0) & (rs_denom_filt == 1e-9)] = 50.0
    data['RSI_filtered'] = rsi_series_filt.fillna(50.0)
    data['Short_span_filtered'] = source_price_for_signals.ewm(span=p_macd_short, adjust=False).mean()
    data['Long_span_filtered'] = source_price_for_signals.ewm(span=p_macd_long, adjust=False).mean()
    data['MACD_filtered'] = data['Short_span_filtered'] - data['Long_span_filtered']
    data['Signal_line_filtered'] = data['MACD_filtered'].ewm(span=p_macd_signal, adjust=False).mean()
    data['Signal_val'] = 0.0
    data.loc[data['MACD_filtered'] > data['Signal_line_filtered'], 'Signal_val'] = 1.0; data.loc[data['MACD_filtered'] < data['Signal_line_filtered'], 'Signal_val'] = -1.0
    data['Position_filtered'] = data['Signal_val'].diff().fillna(0.0)
    start_raw_price = data_close_series.iloc[0] if not data_close_series.empty else 0.0
    end_raw_price = data_close_series.iloc[-1] if not data_close_series.empty else 0.0
    buyhold_profit_amount = 0.0; buyhold_return_percent = 0.0
    if start_raw_price > 0 and pd.notna(start_raw_price) and pd.notna(end_raw_price) and initial_cash > 0 :
        init_qty = initial_cash // start_raw_price; buyhold_profit_amount = (end_raw_price - start_raw_price) * init_qty
        buyhold_return_percent = ((end_raw_price - start_raw_price) / start_raw_price) * 100 if start_raw_price !=0 else 0
    loop_positions_triggers = data['Position_filtered'].squeeze(); loop_rsi_conditions = data['RSI_filtered'].squeeze(); loop_raw_prices_execution = data_close_series
    cash, shares, entry_price_raw = float(initial_cash), 0, 0.0
    trade_log, portfolio_values_raw = [], [cash]
    daily_returns_for_portfolio = []
    risk_free_rate = 0.05
    for i in range(len(data)):
        current_raw_price = loop_raw_prices_execution.iloc[i]
        portfolio_prev_day = portfolio_values_raw[-1]
        if pd.isna(current_raw_price): portfolio_values_raw.append(portfolio_prev_day); daily_returns_for_portfolio.append(0.0) if i > 0 else None; continue
        current_raw_price = float(current_raw_price)
        position_signal = loop_positions_triggers.iloc[i]; rsi_condition_val = loop_rsi_conditions.iloc[i]
        sl_triggered = shares > 0 and entry_price_raw > 0 and current_raw_price < (entry_price_raw * 0.95)
        tp_triggered = shares > 0 and entry_price_raw > 0 and current_raw_price > (entry_price_raw * 1.10)
        was_last_trade_buy = len(trade_log) > 0 and "BUY" in trade_log[-1][1]
        if sl_triggered and was_last_trade_buy: cash += shares*current_raw_price; trade_log.append((data.index[i],"SELL (S)",current_raw_price,shares)); shares,entry_price_raw=0,0.0
        elif tp_triggered and was_last_trade_buy: cash += shares*current_raw_price; trade_log.append((data.index[i],"SELL (T)",current_raw_price,shares)); shares,entry_price_raw=0,0.0
        elif position_signal==2.0 and rsi_condition_val >0 and shares==0 and current_raw_price >0:
            s_buy=int(cash//current_raw_price)
            if s_buy>0: cash-=s_buy*current_raw_price; trade_log.append((data.index[i],"BUY",current_raw_price,s_buy)); entry_price_raw,shares=current_raw_price,s_buy
        elif position_signal==-2.0 and rsi_condition_val >0 and shares >0: cash += shares*current_raw_price; trade_log.append((data.index[i],"SELL",current_raw_price,shares)); shares,entry_price_raw=0,0.0
        portfolio_eod=float(cash+shares*current_raw_price); portfolio_values_raw.append(portfolio_eod)
        if i > 0 and portfolio_prev_day!=0: daily_returns_for_portfolio.append((portfolio_eod-portfolio_prev_day)/portfolio_prev_day)
        elif i > 0: daily_returns_for_portfolio.append(0.0)
    final_value_raw = portfolio_values_raw[-1]; profit_loss_raw = final_value_raw - initial_cash
    daily_returns_series = pd.Series(daily_returns_for_portfolio)
    excess_returns = daily_returns_series - (risk_free_rate / 252)
    sharpe_ratio_raw = 0.0
    if len(excess_returns.dropna()) >= 2 and not pd.isna(excess_returns.std()) and abs(excess_returns.std()) > 1e-9: sharpe_ratio_raw = excess_returns.mean() / excess_returns.std() * np.sqrt(252)
    total_return_percent_strat = (profit_loss_raw / initial_cash) * 100 if initial_cash != 0 else 0
    if show_plots:
        # ... (Plotting logic - condensed for brevity) ...
        trade_df = pd.DataFrame(trade_log, columns=['Date','Action','Price','Shares'])
        plt.figure(figsize=(16,10)); plt.subplot(2,1,1)
        plt.plot(data.index, data_close_series, label="Raw Close", c="k", lw=1.5)
        plt.plot(data.index, kalman_close_series, label="Kalman (Signals)", c="b", alpha=0.7, ls='--')
        if not trade_df.empty: b_tr=trade_df[trade_df['Action']=="BUY"];s_tr_gen=trade_df[trade_df['Action'].str.startswith("SELL")]; plt.scatter(b_tr['Date'],b_tr['Price'],label="B",marker="^",c="g",s=80,zorder=5); plt.scatter(s_tr_gen['Date'],s_tr_gen['Price'],label="S",marker="v",c="r",s=80,zorder=5); sl_tr=trade_df[trade_df['Action']=="SELL (S)"]; tp_tr=trade_df[trade_df['Action']=="SELL (T)"]; plt.scatter(sl_tr['Date'],sl_tr['Price'],label="SL",marker="v",facecolors='none',edgecolors="orange",s=100,zorder=6,lw=1.5); plt.scatter(tp_tr['Date'],tp_tr['Price'],label="TP",marker="v",facecolors='none',edgecolors="purple",s=100,zorder=6,lw=1.5)
        title_str = (f"{ticker} (KF={kalman_qr_factor:.3f}, MACD={p_macd_short},{p_macd_long},{p_macd_signal}) Strat Ret: {total_return_percent_strat:.1f}%, B&H Ret: {buyhold_return_percent:.1f}%")
        plt.title(title_str); plt.legend(); plt.grid(True); plt.subplot(2,1,2); plt.plot(data.index,data['MACD_filtered'],label="MACD_F",c="g");plt.plot(data.index,data['Signal_line_filtered'],label="Sig_F",c="r")
        if not trade_df.empty: buy_dates_macd = trade_df[trade_df['Action']=="BUY"]['Date'].map(lambda x: data.index.asof(x) if pd.notna(x) else pd.NaT).dropna().unique(); sell_dates_macd = trade_df[trade_df['Action']=="SELL"]['Date'].map(lambda x: data.index.asof(x) if pd.notna(x) else pd.NaT).dropna().unique(); plt.scatter(buy_dates_macd, data.loc[buy_dates_macd,'MACD_filtered'].squeeze(),label="B_Sig",marker="^",c="g",s=80,zorder=5) if len(buy_dates_macd)>0 else None; plt.scatter(sell_dates_macd, data.loc[sell_dates_macd,'MACD_filtered'].squeeze(),label="S_Sig",marker="v",c="r",s=80,zorder=5) if len(sell_dates_macd)>0 else None
        plt.title("Filtered MACD (Signal Gen)");plt.legend();plt.grid(True);plt.tight_layout();plt.show()
    return {"Ticker":ticker, "Final Value":final_value_raw, "Net Profit/Loss":profit_loss_raw,"Sharpe Ratio":sharpe_ratio_raw, "Params": {"KF_factor": kalman_qr_factor, "MACD_S": p_macd_short, "MACD_L": p_macd_long, "MACD_Sig": p_macd_signal},"Strategy Total Return %": total_return_percent_strat, "Num Trades": len(trade_log),"Buy and Hold Profit Amount": buyhold_profit_amount, "Buy and Hold Return %": buyhold_return_percent }

# --- Global variables
IT_STOCKS_INDIA = ["ADANIPORTS.NS", "ASIANPAINT.NS", "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS",
    "BAJAJFINSV.NS", "BPCL.NS", "BHARTIARTL.NS", "BRITANNIA.NS", "CIPLA.NS",
    "COALINDIA.NS", "DIVISLAB.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS",
    "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS", "HINDALCO.NS",
    "HINDUNILVR.NS", "ICICIBANK.NS", "ITC.NS", "INDUSINDBK.NS", "INFY.NS",
    "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS",
    "NTPC.NS", "NESTLEIND.NS", "ONGC.NS", "POWERGRID.NS", "RELIANCE.NS",
    "SBILIFE.NS", "SHREECEM.NS", "SBIN.NS", "SUNPHARMA.NS", "TCS.NS",
    "TATACONSUM.NS", "TATAMOTORS.NS", "TATASTEEL.NS", "TECHM.NS", "TITAN.NS",
    "ULTRACEMCO.NS", "UPL.NS", "WIPRO.NS", "ZEEL.NS", "HDFCBANK.NS"]x
OPTIMIZATION_TICKERS_MASTER_LIST = IT_STOCKS_INDIA
IN_SAMPLE_START_DATE = "2020-01-01"; IN_SAMPLE_END_DATE = "2021-12-31"
OUT_OF_SAMPLE_START_DATE = "2022-01-01"; OUT_OF_SAMPLE_END_DATE = "2023-12-31"
INITIAL_CASH_GLOBAL = 100000
_current_ticker_for_opt = None

# --- Objective Function
def objective_function_single_ticker(params_array):
    global _current_ticker_for_opt
    kalman_qr_factor, p_short, p_long, p_signal = params_array

    # Stricter enforcement of conceptual bounds for Nelder-Mead
    if not (1e-5 <= kalman_qr_factor <= 1.5): return 1e10
    if not (5.0 <= p_short <= 25.0): return 1e10
    if not (20.0 <= p_long <= 60.0): return 1e10
    if not (5.0 <= p_signal <= 25.0): return 1e10

    s_span = int(round(p_short)); l_span = int(round(p_long)); sig_span = int(round(p_signal))

    # Enforce practical/logical parameter constraints post-rounding
    if not (1e-6 <= kalman_qr_factor <= 5.0): return 1e9
    if not (3 <= s_span <= 30): return 1e9
    if not (l_span > s_span and l_span >= s_span + 5 and l_span <= 100): return 1e9
    if not (3 <= sig_span <= 30): return 1e9

    result = sma_trading_strategy(
        _current_ticker_for_opt, IN_SAMPLE_START_DATE, IN_SAMPLE_END_DATE, INITIAL_CASH_GLOBAL,
        kalman_qr_factor=kalman_qr_factor, macd_short_span=s_span, macd_long_span=l_span, macd_signal_span=sig_span,
        show_plots=False )
    if result and pd.notna(result['Sharpe Ratio']):
        sharpe = result['Sharpe Ratio']
        if pd.isna(sharpe) or sharpe < -5 : return 1e9
        return -sharpe
    else:
        return 1e9

# --- Main Execution Block with Nelder-Mead ---
if __name__ == '__main__':
    all_stocks_optimized_results = []
    print("Starting INDIVIDUAL STOCK Multi-Parameter Optimization (Kalman + MACD) using Nelder-Mead...\n")

    for ticker_to_optimize in OPTIMIZATION_TICKERS_MASTER_LIST:
        print(f"--- Optimizing for Ticker: {ticker_to_optimize} ---")
        _current_ticker_for_opt = ticker_to_optimize

        initial_guess_multi = [0.01, 12.0, 26.0, 9.0]
        # Bounds are not passed to Nelder-Mead in the call, but enforced by objective_function

        print(f"  In-Sample Period: {IN_SAMPLE_START_DATE} to {IN_SAMPLE_END_DATE}")
        # --- Using Nelder-Mead ---
        optimization_result = minimize(
            objective_function_single_ticker,
            initial_guess_multi,
            method='Nelder-Mead',
            options={'disp': False, 'maxiter': 100, # Increase maxiter for more thorough search
                     'xatol': 0.01, 'fatol': 1e-3,   # Tolerances for Nelder-Mead
                     'adaptive': True} # Helps Nelder-Mead adapt its simplex
        )

        stock_opt_data = {"ticker": ticker_to_optimize, "in_sample": {}, "out_of_sample": {}, "optimization_success": optimization_result.success}

        if optimization_result.success or optimization_result.status == 0: # Status 0 can also be acceptable for NM
            opt_params_arr = optimization_result.x
            opt_kf, opt_s_raw, opt_l_raw, opt_sig_raw = opt_params_arr
            # Sanitize parameters after optimization for final use
            opt_s = int(round(opt_s_raw)); opt_l = int(round(opt_l_raw));
            opt_l = max(opt_s+1, opt_l, int(opt_s * 1.5)) # Ensure long is meaningfully larger and apply lower bound
            opt_sig = int(round(opt_sig_raw))
            # Re-check bounds for sanitized parameters before storing/using
            opt_kf = max(1e-5, min(1.5, opt_kf))
            opt_s = max(3, min(30, opt_s))
            opt_l = max(opt_s + 5, min(100, opt_l))
            opt_sig = max(3, min(30, opt_sig))

            final_optimal_params = {"KF_factor": opt_kf, "MACD_S": opt_s, "MACD_L": opt_l, "MACD_Sig": opt_sig}
            stock_opt_data["optimal_params"] = final_optimal_params
            print(f"  Optimal Parameters for {ticker_to_optimize}: KF={opt_kf:.4f}, MACD=({opt_s},{opt_l},{opt_sig})")
            print(f"  Optimizer Message: {optimization_result.message} (Value: {optimization_result.fun:.4f})")

            # ... (In-sample and Out-of-sample evaluation as before) ...
            print(f"  Evaluating optimal params on In-Sample data for {ticker_to_optimize}...")
            is_result_optimal = sma_trading_strategy(
                ticker_to_optimize, IN_SAMPLE_START_DATE, IN_SAMPLE_END_DATE, INITIAL_CASH_GLOBAL,
                kalman_qr_factor=opt_kf, macd_short_span=opt_s, macd_long_span=opt_l, macd_signal_span=opt_sig,
                show_plots=False )
            if is_result_optimal: stock_opt_data["in_sample"] = is_result_optimal; print(f"    In-Sample (Optimal): Sharpe={is_result_optimal['Sharpe Ratio']:.4f}, Strat Ret%={is_result_optimal['Strategy Total Return %']:.2f}%, B&H Ret%={is_result_optimal['Buy and Hold Return %']:.2f}%")
            else: stock_opt_data["in_sample"]["Sharpe Ratio"] = -optimization_result.fun if hasattr(optimization_result, 'fun') else np.nan; stock_opt_data["in_sample"]["error"] = "Failed IS results"
        else: # Optimization failed
            print(f"  Optimization FAILED for {ticker_to_optimize}. Message: {optimization_result.message}")
            stock_opt_data["optimal_params"] = None; stock_opt_data["in_sample"]["error"] = optimization_result.message
            if hasattr(optimization_result, 'x') and optimization_result.x is not None:
                 kf, s, l, sig = optimization_result.x; s_r, l_r, sig_r = int(round(s)), int(round(l)), int(round(sig)); l_r = max(s_r+1, l_r)
                 stock_opt_data["attempted_params_on_fail"] = {"KF_factor": kf, "MACD_S":s_r, "MACD_L":l_r, "MACD_Sig": sig_r}

        if stock_opt_data.get("optimal_params"):
            print(f"  Validating optimal params on Out-of-Sample data for {ticker_to_optimize}...")
            oos_result = sma_trading_strategy( ticker_to_optimize, OUT_OF_SAMPLE_START_DATE, OUT_OF_SAMPLE_END_DATE, INITIAL_CASH_GLOBAL,
                kalman_qr_factor=stock_opt_data["optimal_params"]["KF_factor"], macd_short_span=stock_opt_data["optimal_params"]["MACD_S"],
                macd_long_span=stock_opt_data["optimal_params"]["MACD_L"], macd_signal_span=stock_opt_data["optimal_params"]["MACD_Sig"],
                show_plots= (len(OPTIMIZATION_TICKERS_MASTER_LIST) <= 3) )
            if oos_result: stock_opt_data["out_of_sample"] = oos_result; print(f"    Out-of-Sample: Sharpe={oos_result['Sharpe Ratio']:.4f}, Strat Ret%={oos_result['Strategy Total Return %']:.2f}%, B&H Ret%={oos_result['Buy and Hold Return %']:.2f}%")
        all_stocks_optimized_results.append(stock_opt_data)
        print("-" * 70)

    # --- Final Summary
    print("\n\n--- FINAL SUMMARY ACROSS ALL OPTIMIZED STOCKS (Out-of-Sample Performance) ---")
    valid_oos_entries = [res for res in all_stocks_optimized_results if res.get("optimal_params") and res.get("out_of_sample")]
    if valid_oos_entries:
        print(f"{'Ticker':<15} | {'OOS Strat Shrp':>14} | {'OOS Strat P/L':>13} | {'OOS Strat Ret%':>14} | {'OOS B&H P/L':>12} | {'OOS B&H Ret%':>12} | Trades | Opt Params (KF,S,L,Sig)")
        print("-" * 120)
        for entry in valid_oos_entries:
            res = entry["out_of_sample"]; opt_p = entry["optimal_params"]
            params_str = f"({opt_p.get('KF_factor',0):.3f},{opt_p.get('MACD_S',0)},{opt_p.get('MACD_L',0)},{opt_p.get('MACD_Sig',0)})"
            print(f"{res['Ticker']:<15} | {res.get('Sharpe Ratio', float('nan')):>14.4f} | {res.get('Net Profit/Loss', 0):>13.2f} | {res.get('Strategy Total Return %', 0):>14.2f}% | {res.get('Buy and Hold Profit Amount',0):>12.2f} | {res.get('Buy and Hold Return %',0):>12.2f}% | {res.get('Num Trades',0):<6} | {params_str}")
        oos_strat_sharpes = [r['out_of_sample']['Sharpe Ratio'] for r in valid_oos_entries if pd.notna(r['out_of_sample'].get('Sharpe Ratio'))]
        oos_strat_profits = [r['out_of_sample']['Net Profit/Loss'] for r in valid_oos_entries]
        oos_strat_returns_pct = [r['out_of_sample']['Strategy Total Return %'] for r in valid_oos_entries]
        oos_bh_profits = [r['out_of_sample']['Buy and Hold Profit Amount'] for r in valid_oos_entries]
        oos_bh_returns_pct = [r['out_of_sample']['Buy and Hold Return %'] for r in valid_oos_entries]
        oos_num_trades = [r['out_of_sample']['Num Trades'] for r in valid_oos_entries]
        print("-" * 120); print("Basket Averages (Out-of-Sample, using individually optimized params):")
        print("  Strategy:");
        if oos_strat_sharpes: print(f"    Avg Sharpe: {np.mean(oos_strat_sharpes):.4f}")
        if oos_strat_profits: print(f"    Avg Net P/L: {np.mean(oos_strat_profits):.2f}")
        if oos_strat_returns_pct: print(f"    Avg Return %: {np.mean(oos_strat_returns_pct):.2f}%")
        if oos_num_trades: print(f"    Avg Num Trades: {np.mean(oos_num_trades):.1f}")
        print("  Buy & Hold:");
        if oos_bh_profits: print(f"    Avg B&H Profit Amount: {np.mean(oos_bh_profits):.2f}")
        if oos_bh_returns_pct: print(f"    Avg B&H Return %: {np.mean(oos_bh_returns_pct):.2f}%")
    else: print("No valid out-of-sample results obtained for summary.")

Starting INDIVIDUAL STOCK Multi-Parameter Optimization (Kalman + MACD) using Nelder-Mead...

--- Optimizing for Ticker: ADANIPORTS.NS ---
  In-Sample Period: 2020-01-01 to 2021-12-31
  Optimal Parameters for ADANIPORTS.NS: KF=0.0101, MACD=(12,26,9)
  Optimizer Message: Optimization terminated successfully. (Value: -0.6558)
  Evaluating optimal params on In-Sample data for ADANIPORTS.NS...
    In-Sample (Optimal): Sharpe=0.6558, Strat Ret%=34.77%, B&H Ret%=95.26%
  Validating optimal params on Out-of-Sample data for ADANIPORTS.NS...
    Out-of-Sample: Sharpe=1.4912, Strat Ret%=68.84%, B&H Ret%=40.97%
----------------------------------------------------------------------
--- Optimizing for Ticker: ASIANPAINT.NS ---
  In-Sample Period: 2020-01-01 to 2021-12-31
  Optimal Parameters for ASIANPAINT.NS: KF=0.0103, MACD=(12,26,9)
  Optimizer Message: Optimization terminated successfully. (Value: -0.9030)
  Evaluating optimal params on In-Sample data for ASIANPAINT.NS...
    In-Sample (Optimal)